# Pelatihan Model Regresi Estimasi Harga Laptop Gaming (GamerGear)

Notebook ini digunakan untuk melatih model machine learning regresi guna mengestimasi harga wajar laptop gaming berdasarkan spesifikasinya, menghitung skor value penawaran harga, serta memperbarui database `gamergear.db`.

### Sel 1: Ambil Data dari Database
Tarik data bersih dari tabel `laptops` yang sudah disimpan sebelumnya menggunakan `sqlite3` dan `pandas`.

In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/gamergear.db')
df = pd.read_sql_query("SELECT * FROM laptops", conn)
df.head()

,id,nama,merek,cpu,gpu,ram_gb,ssd_gb,layar_inci,refresh_rate,harga_listing,harga_prediksi,skor_value,tier_performa
0,1,Lenovo LOQ 83DV007GIN Gaming Laptop,Lenovo,13th Gen Intel Core i5 13450HX,6 GB NVIDIA GeForce RTX 4050,16,512,15.6,144,25080000,21399028,-0.172016,2.5
1,2,HP Victus 15-fb3125AX Gaming Laptop,HP,7th Gen AMD Ryzen 7 7445H,NVIDIA GeForce RTX 4050,16,512,15.0,144,15673860,23215960,0.324867,3.0
2,3,HP Victus 15-fa2192TX Gaming Laptop,HP,13th Gen Intel Core i5 13420H,4 GB NVIDIA GeForce RTX 3050,16,512,15.0,144,15008100,18611958,0.193631,2.0
3,4,Lenovo LOQ Essential 83S000JCIN Gaming Laptop,Lenovo,7th Gen AMD Ryzen 5 7535HS,6 GB NVIDIA GeForce RTX 3050,16,512,15.6,144,17098100,19237316,0.111201,2.0
4,5,Lenovo LOQ 2023 Gaming Laptop,Lenovo,13th Gen Intel Core i5 13420H,8 GB NVIDIA GeForce RTX 4060,8,512,15.6,144,17099810,19591301,0.127173,3.0


### Sel 2: Pisahkan Fitur (X) dan Target (y)
Tentukan kolom spesifikasi sebagai fitur masukan dan `harga_listing` sebagai target prediksi. Gunakan `pd.get_dummies` jika kolom merek masih berupa teks.

In [5]:
from sklearn.model_selection import train_test_split

# Sesuaikan daftar kolom dengan nama kolom di tabel laptops milikmu
fitur = ['ram_gb', 'ssd_gb', 'layar_inci', 'refresh_rate', 'tier_performa'] 
X = df[fitur]
y = df['harga_listing']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Sel 3: Latih & Bandingkan Algoritma
Latih kandidat model (Linear Regression, Random Forest, Gradient Boosting) dan ukur akurasinya dengan MAE, RMSE, dan $R^2$.

In [6]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

models = {
    "Ridge": Ridge(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

for nama, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"{nama} -> MAE: {mean_absolute_error(y_test, pred):,.0f} | R2: {r2_score(y_test, pred):.3f}")

Ridge -> MAE: 6,316,865 | R2: 0.712
Random Forest -> MAE: 5,574,328 | R2: 0.646
Gradient Boosting -> MAE: 5,741,429 | R2: 0.665


### Sel 4: Hitung Skor Value & Update Database
Pilih model dengan skor $R^2$ tertinggi (misalnya Random Forest), prediksi seluruh data, hitung `skor_value`, lalu simpan kembali ke SQLite dan ekspor model ke folder `model/`.

In [7]:
import os
import joblib

# Pastikan direktori model tersedia
os.makedirs('../model', exist_ok=True)

best_model = models["Random Forest"]
df['harga_prediksi'] = best_model.predict(X).astype(int)
df['skor_value'] = (df['harga_prediksi'] - df['harga_listing']) / df['harga_prediksi']

# Perbarui tabel laptops di database
df.to_sql('laptops', conn, if_exists='replace', index=False)
conn.close()

# Simpan model ke file .pkl
joblib.dump(best_model, '../model/laptop_price_model.pkl')
print("Model regresi disimpan dan database diperbarui!")

Model regresi disimpan dan database diperbarui!
